# ParticleNet-style graph network: quark/gluon classification

This notebook trains a **ParticleNet-style graph network** on every constituent of each truth-matched jet.
It follows the common preparation lesson and saves a self-describing model bundle.

EdgeConv learns from local constituent neighborhoods, treating a jet as a particle cloud rather than a fixed vector.

Set `QG_RUN_MODE=full` before launching Jupyter for the larger configuration. Quick
mode is the default. Both automatically use CUDA when it is available.


## 1. Environment and device

Run `./setup_student_env.sh` once before this lesson. This check confirms that the selected Jupyter kernel is the student-owned henv; CPU remains a supported fallback.


In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'sklearn', 'torch', 'tqdm']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. From a terminal in this directory run "
        "./setup_student_env.sh (or use --current inside an existing henv), "
        "restart Jupyter from that henv, and select its registered kernel."
    )

import json, os, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm
import qg_constituent_ml as qg

DEVICE = qg.choose_device()
RUN_MODE = os.getenv('QG_RUN_MODE', 'quick')
SOURCE = Path(os.getenv('QG_INPUT_PATH', 'data/inclusive_jets.parquet'))
print(f'PyTorch {torch.__version__}; built with CUDA {torch.version.cuda}')
print(f'device={DEVICE}' + (f'; GPU={torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print(f'run mode={RUN_MODE}; source={SOURCE}')


## 2. Current data and model

Preparation is fingerprint-aware: changing the generated Parquet file creates a new prepared-data namespace automatically.


In [ ]:
prepared = qg.prepare_dataset(SOURCE)
manifest = qg.load_manifest(prepared)
MODEL_CONFIG = qg.default_config('particlenet', RUN_MODE)
model = qg.create_model('particlenet', config=MODEL_CONFIG)
print(json.dumps({k: manifest[k] for k in ('source_sha256','n_jets','n_constituents','split_counts','class_counts')}, indent=2))
print(model)
print(f'parameters={sum(p.numel() for p in model.parameters()):,}; config={MODEL_CONFIG}')


### Canonical implementation

The reusable class lives in `qg_constituent_ml.py` so saved models can be reconstructed later. Its source is displayed here to keep the architecture visible.


In [ ]:
import inspect
from IPython.display import Code, display
display(Code(inspect.getsource(qg.model_classes()['particlenet']), language='python'))


## 3. Invariance checks

Padding must not affect a prediction. Reordering constituents must not change an unordered-set classifier.


In [ ]:
loaders = qg.make_loaders(prepared, 'particlenet', RUN_MODE)
batch = next(iter(loaders[2]))
model.eval()
with torch.no_grad():
    base = model(batch['features'][:4], batch['coords'][:4], batch['mask'][:4])
    order = torch.randperm(batch['features'].shape[1])
    permuted = model(batch['features'][:4, order], batch['coords'][:4, order], batch['mask'][:4, order])
    padded_f = torch.nn.functional.pad(batch['features'][:4], (0,0,0,3))
    padded_c = torch.nn.functional.pad(batch['coords'][:4], (0,0,0,3))
    padded_m = torch.nn.functional.pad(batch['mask'][:4], (0,3))
    padded = model(padded_f, padded_c, padded_m)
assert torch.allclose(base, permuted, atol=2e-5)
assert torch.allclose(base, padded, atol=2e-5)
print('Passed permutation and padding invariance checks.')


## 4. Train and save

Training is balanced, validation/test mixtures are natural, and early stopping restores the best validation checkpoint.


In [ ]:
history, metrics, predictions = qg.train_model(model, loaders, 'particlenet', RUN_MODE, DEVICE)
bundle = qg.save_model_bundle(model, 'particlenet', RUN_MODE, MODEL_CONFIG, prepared,
                              history, metrics, predictions)
print(json.dumps(metrics, indent=2))
print(f'Saved model bundle: {bundle}')


In [ ]:
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 2, figsize=(11,4))
axes[0].plot([x['epoch'] for x in history], [x['train_loss'] for x in history], marker='o')
axes[0].set(xlabel='epoch', ylabel='BCE loss', title='Training history')
fpr,tpr,_=roc_curve(predictions['labels'], predictions['scores'])
axes[1].plot(tpr,1/np.clip(fpr,1e-3,None),label=f"AUC={metrics['roc_auc']:.3f}")
axes[1].set(xlabel='quark efficiency',ylabel='gluon rejection',yscale='log',title='Held-out performance')
axes[1].legend(); plt.tight_layout(); plt.show()


## 5. What this result means

Compare architectures only through the evaluation notebook, which enforces matching dataset and split fingerprints. A larger network on this small sample is not automatically a better physics model.
